# Speech-to-Speech end-to-end smoke test (Colab)
Whisper-small → MarianMT base → XTTS-v2 → ECAPA, únicamente con tres muestras FLEURS DEV.

In [ ]:
# Celda autosuficiente de instalación para un runtime Colab nuevo; ejecutar antes de importar torch.
%pip install -q 'torch==2.8.0' 'torchaudio==2.8.0' 'coqui-tts==0.27.5' 'transformers==4.57.3' 'numpy==2.4.6' 'speechbrain==1.1.1' 'soundfile==0.14.0' 'scipy==1.17.1' 'datasets==3.6.0' 'pyarrow==18.1.0' 'pandas==2.3.3' 'pyyaml==6.0.3' 'jiwer==3.1.0' 'sentencepiece==0.2.2' 'sacremoses==0.1.1' 'accelerate==1.12.0'
import os,sys,json,subprocess
from pathlib import Path
REPO_URL='https://github.com/Setrox1250/Proyecto-IA-Traductor.git'; ROOT=Path('/content/Proyecto-IA-Traductor')
if not ROOT.exists(): subprocess.run(['git','clone',REPO_URL,str(ROOT)],check=True)
os.chdir(ROOT); sys.path.insert(0,str(ROOT))
import pandas as pd
from src.dataset.fleurs import stream_fleurs, take_samples, build_fleurs_manifest
from src.dataset.manifest import load_fleurs_english_references, apply_fleurs_references, assign_splits
from src.pipeline.runner import process_audio
manifest_path=ROOT/'data/processed/manifests/evaluation_v1.csv'
if manifest_path.exists(): manifest=pd.read_csv(manifest_path)
else:
    manifest=build_fleurs_manifest(take_samples(stream_fleurs(),n=20,seed=42),audio_dir=ROOT/'data/raw/audio')
    manifest=assign_splits(apply_fleurs_references(manifest,load_fleurs_english_references()),seed=42); manifest_path.parent.mkdir(parents=True,exist_ok=True); manifest.to_csv(manifest_path,index=False)
dev=manifest.loc[manifest['split']=='dev'].head(3); assert len(dev)==3
results=[]
for _, row in dev.iterrows():
    run_dir=ROOT/'outputs/smoke_tests/pipeline'/row['sample_id']; run_dir.mkdir(parents=True,exist_ok=True)
    (run_dir/'reference_transcription.txt').write_text(str(row['transcription_es']),encoding='utf-8')
    result=process_audio(str(ROOT/row['audio_path']),str(run_dir)); result['reference_es']=str(row['transcription_es']); result['reference_en']=str(row['translation_en']); results.append(result)
for result in results: print(json.dumps({'sample_id':result['sample_id'],'ground_truth_es':result['reference_es'],'whisper_prediction':result['transcription_es'],'marian_translation':result['translation_en'],'reference_en':result['reference_en'],'cosine_similarity':result['speaker_similarity'],'timings':result['timings'],'rtf_total':result['rtf_total']},ensure_ascii=False,indent=2))

In [ ]:
from IPython.display import Audio, display
for result in results:
    print(result['sample_id'])
    display(Audio(str(ROOT/'outputs/smoke_tests/pipeline'/result['sample_id']/'input.wav')))
    display(Audio(result['audio']['output_path']))